In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import math
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
from preprocessing import floor_tf, abs_tf, fix_year_transformer, fix_paint_transformer, fix_mpg_transformer, FixPreviousOwners


In [4]:
train= pd.read_csv("train.csv").copy() #importing the dataset
test=pd.read_csv("test.csv").copy()

In [ ]:
year_col      = ["year"]
mpg_col       = ["mpg"]
paint_col     = ["paintQuality%"]
mileage_col   = ["mileage"]
tax_col       = ["tax"]
engine_col    = ["engineSize"]
prevown_col   = ["previousOwners"]

# YEAR PIPELINE
year_pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("floor", floor_tf),                
    ("clip",  fix_year_transformer),    # 2020 is the maximum currently defined
    ("scale", StandardScaler())
])

mpg_pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("abs",   abs_tf),
    ("clip",  fix_mpg_transformer),     # range must be 10-200
    ("scale", StandardScaler())
])

paint_pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("abs",   abs_tf),
    ("clip",  fix_paint_transformer),   # range 0-100 - because it is percentage
    ("scale", StandardScaler())
])

mileage_pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("abs",   abs_tf),
    ("scale", StandardScaler())
])

tax_pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("abs",   abs_tf),
    ("scale", StandardScaler())
])

engine_pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("abs",   abs_tf),
    ("scale", StandardScaler())
])

previousowners_pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="mean")),  
    ("abs",   abs_tf),
    ("round", FunctionTransformer(lambda X: np.rint(np.asarray(X, dtype=float)),
                                  feature_names_out="one-to-one")),
    ("scale", StandardScaler())
])

In [6]:
numeric_feat = ColumnTransformer(transformers=[
    ("year",      year_pipe,    year_col),
    ("mpg",       mpg_pipe,     mpg_col),
    ("paint",     paint_pipe,   paint_col),
    ("mileage",   mileage_pipe, mileage_col),
    ("tax",       tax_pipe,     tax_col),
    ("engine",    engine_pipe,  engine_col),
    ("prevown",   previousowners_pipe, prevown_col),
], remainder="drop", verbose_feature_names_out=False)

In [7]:
fix_prev_owners = FixPreviousOwners(max_year=2020, mileage_threshold=15000, age_threshold=2)
preprocess_numeric_owners = Pipeline([
    ("fix_prevOwners", fix_prev_owners),
    ("by_column",      numeric_feat),
])